In [1]:
# configuración
# Imports, semilla aleatoria y carga de archivos desde disco.

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, roc_auc_score

# GAP: semilla no reportada en el paper — convención estándar
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Ruta base del dataset IoT-23
BASE_DIR = (Path.home() / "Downloads" / "iot_23_datasets_small" / "opt" / "Malware-Project" / "BigDataset" / "IoTScenarios")

def get_labeled_path(capture_id):
    return BASE_DIR / f"CTU-IoT-Malware-Capture-{capture_id}" / "bro" / "conn.log.labeled"

# Escenario 2: train = D01+D02+D03 | test = D06, D09, D13, D18, D20
DATASETS = {
    "D01": {"capture": "1-1",  "malware": "Hide and Seek", "role": "train"},
    "D02": {"capture": "3-1",  "malware": "Muhstik",       "role": "train"},
    "D03": {"capture": "7-1",  "malware": "Linux.Mirai",   "role": "train"},
    "D06": {"capture": "17-1", "malware": "Kenjiro",       "role": "test"},
    "D09": {"capture": "20-1", "malware": "Kenjiro",       "role": "test"},
    "D13": {"capture": "34-1", "malware": "IRCBot",        "role": "test"},
    "D18": {"capture": "43-1", "malware": "Mirai",         "role": "test"},
    "D20": {"capture": "48-1", "malware": "Hide and Seek", "role": "test"},
}

def read_conn_log_labeled(filepath, max_rows=None):
    """Lee conn.log.labeled de IoT-23 y extrae columnas label/detailed-label."""
    filepath = Path(filepath)
    col_names = None
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('#fields'):
                col_names = line.strip().split('\t')[1:]
                break
    if col_names is None:
        raise ValueError(f"No se encontró #fields en {filepath.name}")

    df = pd.read_csv(
        filepath, sep='\t', comment='#', header=None, names=col_names,
        na_values=['-', '(empty)', ''], nrows=max_rows, low_memory=False
    )

    # La última columna contiene tunnel_parents + label + detailed-label separados por espacios
    last_col = df.columns[-1]
    split_cols = df[last_col].astype(str).str.split(r'\s+', expand=True)
    if split_cols.shape[1] >= 3:
        df['tunnel_parents'] = split_cols[0]
        df['label']          = split_cols[1]
        df['detailed-label'] = split_cols[2]
    elif split_cols.shape[1] == 2:
        df['label']          = split_cols[0]
        df['detailed-label'] = split_cols[1]
    else:
        df['label'] = split_cols[0]
    if last_col not in ['label', 'detailed-label', 'tunnel_parents']:
        df.drop(columns=[last_col], inplace=True)
    return df

# Verificar disponibilidad de archivos
print(f"{'Dataset':<6} {'Rol':<6} {'Malware':<16} {'Estado'}")
print("-" * 55)
for ds_id, info in DATASETS.items():
    path = get_labeled_path(info["capture"])
    exists = path.exists()
    size_mb = round(path.stat().st_size / 1024 / 1024, 1) if exists else 0
    status = f"OK  {size_mb} MB" if exists else "NO ENCONTRADO"
    print(f"  {ds_id:<6} {info['role']:<6} {info['malware']:<16} {status}")


Dataset Rol    Malware          Estado
-------------------------------------------------------
  D01    train  Hide and Seek    OK  141.4 MB
  D02    train  Muhstik          OK  23.3 MB
  D03    train  Linux.Mirai      OK  1508.0 MB
  D06    test   Kenjiro          OK  7762.3 MB
  D09    test   Kenjiro          OK  0.4 MB
  D13    test   IRCBot           OK  2.9 MB
  D18    test   Mirai            OK  8929.5 MB
  D20    test   Hide and Seek    OK  507.1 MB


In [2]:
    # preprocesamiento y seleccion: one-hot + MinMaxScaler. El paper no lista las 13 features, se aproximan con las de conn.log

FEATURES_NUMERIC = [
    'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes',
    'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
]
FEATURES_CATEGORICAL = ['proto', 'conn_state', 'service']

MAX_ROWS = 200_000  # ajustar según RAM disponible; None = leer todo

def preprocess(df, fit_encoders=None):
    """Etiqueta binaria, codificación de categóricas y limpieza de NaN."""
    df = df.copy()
    label_raw = df['label'].astype(str).str.strip().str.lower()
    # Excluir flujos Background (ambiguos, no etiquetados con certeza)
    mask = label_raw.isin(['malicious', 'benign'])
    df, label_raw = df[mask].copy(), label_raw[mask]
    y = (label_raw == 'malicious').astype(int).values

    avail_num = [f for f in FEATURES_NUMERIC    if f in df.columns]
    avail_cat = [f for f in FEATURES_CATEGORICAL if f in df.columns]
    X = df[avail_num + avail_cat].copy()

    encoders = fit_encoders.copy() if fit_encoders else {}
    for col in avail_cat:
        X[col] = X[col].astype(str).str.strip()
        if fit_encoders and col in fit_encoders:
            le = fit_encoders[col]
            # Categorías no vistas en train → clase 0
            X[col] = X[col].apply(lambda v: le.transform([v])[0] if v in le.classes_ else 0)
        else:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col])
            encoders[col] = le

    X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    # Features derivadas para aproximar las 13 del paper
    if 'orig_bytes' in X.columns and 'resp_bytes' in X.columns:
        X['bytes_ratio'] = X['resp_bytes'] / (X['orig_bytes'] + 1)
        X['total_bytes'] = X['orig_bytes'] + X['resp_bytes']
    if 'orig_pkts' in X.columns and 'resp_pkts' in X.columns:
        X['pkts_ratio'] = X['resp_pkts'] / (X['orig_pkts'] + 1)

    mask_nan = ~X.isnull().any(axis=1)
    return X[mask_nan].values, y[mask_nan], encoders

# Cargar y concatenar D01 + D02 + D03 (conjunto de entrenamiento)
print("Cargando conjuntos de entrenamiento: D01 + D02 + D03")
X_list, y_list, train_encoders = [], [], None
for ds_id in ["D01", "D02", "D03"]:
    info = DATASETS[ds_id]
    df = read_conn_log_labeled(get_labeled_path(info["capture"]), max_rows=MAX_ROWS)
    X, y, encoders = preprocess(df, fit_encoders=train_encoders)
    if train_encoders is None:
        train_encoders = encoders
    X_list.append(X)
    y_list.append(y)
    print(f"  [{ds_id}] {info['malware']:<16} filas: {len(X):,} "
          f"| Benign: {(y==0).sum():,} | Malicious: {(y==1).sum():,}")

X_train_raw = np.vstack(X_list)
y_train     = np.concatenate(y_list)
N_FEATURES  = X_train_raw.shape[1]

# Split 80/20 (GAP: ratio no especificado en el paper)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.20, random_state=SEED, stratify=y_train
)

# Normalización MinMaxScaler ajustado SOLO sobre el conjunto train
scaler       = MinMaxScaler()
X_tr_scaled  = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)

print(f"\nTrain: {X_tr_scaled.shape} | Val: {X_val_scaled.shape} | Features: {N_FEATURES}")

# Cargar y escalar los conjuntos de test (D06–D20)
# El scaler del train se aplica sin reajuste para evitar data leakage
TEST_IDS = ["D06", "D09", "D13", "D18", "D20"]
test_sets = {}
print("\nConjuntos de test:")
for ds_id in TEST_IDS:
    info = DATASETS[ds_id]
    df = read_conn_log_labeled(get_labeled_path(info["capture"]), max_rows=MAX_ROWS)
    X_raw, y_test, _ = preprocess(df, fit_encoders=train_encoders)
    test_sets[ds_id] = {
        "X_scaled": scaler.transform(X_raw),
        "y":        y_test,
        "malware":  info["malware"],
    }
    print(f"  [{ds_id}] {info['malware']:<16} Benign: {(y_test==0).sum():,} "
          f"| Malicious: {(y_test==1).sum():,}")


Cargando conjuntos de entrenamiento: D01 + D02 + D03
  [D01] Hide and Seek    filas: 44,464 | Benign: 5,546 | Malicious: 38,918
  [D02] Muhstik          filas: 82,159 | Benign: 3,539 | Malicious: 78,620
  [D03] Linux.Mirai      filas: 621 | Benign: 171 | Malicious: 450

Train: (101795, 14) | Val: (25449, 14) | Features: 14

Conjuntos de test:
  [D06] Kenjiro          Benign: 109 | Malicious: 199,891
  [D09] Kenjiro          Benign: 2,316 | Malicious: 8
  [D13] IRCBot           Benign: 1,003 | Malicious: 4,318
  [D18] Mirai            Benign: 26 | Malicious: 40
  [D20] Hide and Seek    Benign: 259 | Malicious: 199,741


In [3]:
# definición y entrenamiento
# Autoencoder Van et al. 2020 (9 capas, capa latente dim=3)
# + Grid Search para DT, kNN y SVM sobre el espacio latente.

def build_autoencoder(input_dim, latent_dim=3):
    """Arquitectura exacta: input→32→24→12→6→[3]→6→12→24→32→output."""
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(32, activation='tanh')(inputs)
    x = layers.Dense(24, activation='tanh')(x)
    x = layers.Dense(12, activation='tanh')(x)
    x = layers.Dense(6,  activation='tanh')(x)
    latent  = layers.Dense(latent_dim, activation='tanh', name='latent')(x)
    x = layers.Dense(6,  activation='tanh')(latent)
    x = layers.Dense(12, activation='tanh')(x)
    x = layers.Dense(24, activation='tanh')(x)
    x = layers.Dense(32, activation='tanh')(x)
    outputs = layers.Dense(input_dim, activation='linear')(x)
    autoencoder = Model(inputs, outputs, name='AE_Van2020')
    encoder     = Model(inputs, latent,  name='Encoder_Van2020')
    return autoencoder, encoder

autoencoder, encoder = build_autoencoder(input_dim=N_FEATURES, latent_dim=3)
# GAP: lr, epochs y batch_size no reportados → valores estándar 2020
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')

# Entrenamiento no supervisado (labels ignorados, como indica el paper)
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=0
)
autoencoder.fit(
    X_tr_scaled, X_tr_scaled,
    validation_data=(X_val_scaled, X_val_scaled),
    epochs=50, batch_size=256,
    callbacks=[early_stop], verbose=1
)

# Proyectar train y val al espacio latente AE-3 (3 dims)
Z_train = encoder.predict(X_tr_scaled,  verbose=0)
Z_val   = encoder.predict(X_val_scaled, verbose=0)
print(f"\nEspacio latente AE-3: Train={Z_train.shape} | Val={Z_val.shape}")

# Grid Search — Decision Tree
print("\n[1/3] Grid Search — Decision Tree")
dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=SEED),
    param_grid={
        'max_depth':         [3, 5, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'criterion':         ['gini', 'entropy'],
    },
    cv=5, scoring='f1', n_jobs=-1
)
dt_gs.fit(Z_train, y_tr)
best_dt = dt_gs.best_estimator_
print(f"  Mejores params: {dt_gs.best_params_} | CV F1: {dt_gs.best_score_:.4f}")

# Grid Search — kNN
print("\n[2/3] Grid Search — kNN")
knn_gs = GridSearchCV(
    KNeighborsClassifier(),
    param_grid={
        'n_neighbors': [3, 5, 7, 10, 15],
        'metric':      ['euclidean', 'manhattan'],
        'weights':     ['uniform', 'distance'],
    },
    cv=5, scoring='f1', n_jobs=-1
)
knn_gs.fit(Z_train, y_tr)
best_knn = knn_gs.best_estimator_
print(f"  Mejores params: {knn_gs.best_params_} | CV F1: {knn_gs.best_score_:.4f}")

# SVM — LinearSVC calibrado
# GAP: el paper no especifica variante ni kernel de SVM.
# LinearSVC es viable a esta escala (~100k muestras); SVC(rbf) sería O(n²).
print("\n[3/3] SVM — LinearSVC calibrado")
best_f1, best_svm, best_C = -1, None, None
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = CalibratedClassifierCV(LinearSVC(C=C, max_iter=2000, random_state=SEED), cv=3)
    clf.fit(Z_train, y_tr)
    f1 = f1_score(y_val, clf.predict(Z_val), pos_label=1, average='binary')
    print(f"  C={C:5.2f} → F1 val: {f1*100:.2f}%")
    if f1 > best_f1:
        best_f1, best_svm, best_C = f1, clf, C
print(f"  Mejor C: {best_C}")

classifiers = {'DT': best_dt, 'kNN': best_knn, 'SVM': best_svm}


Epoch 1/50
398/398 [==============================] - 5s 4ms/step - loss: 0.0052 - val_loss: 0.0041
Epoch 2/50
398/398 [==============================] - 1s 3ms/step - loss: 5.3609e-04 - val_loss: 0.0039
Epoch 3/50
398/398 [==============================] - 1s 3ms/step - loss: 4.5535e-04 - val_loss: 0.0038
Epoch 4/50
398/398 [==============================] - 1s 3ms/step - loss: 3.7592e-04 - val_loss: 0.0037
Epoch 5/50
398/398 [==============================] - 1s 3ms/step - loss: 2.7477e-04 - val_loss: 0.0036
Epoch 6/50
398/398 [==============================] - 1s 3ms/step - loss: 2.1966e-04 - val_loss: 0.0036
Epoch 7/50
398/398 [==============================] - 1s 3ms/step - loss: 1.8601e-04 - val_loss: 0.0035
Epoch 8/50
398/398 [==============================] - 1s 3ms/step - loss: 1.5560e-04 - val_loss: 0.0035
Epoch 9/50
398/398 [==============================] - 1s 3ms/step - loss: 1.2882e-04 - val_loss: 0.0035
Epoch 10/50
398/398 [==============================] - 1s 3ms/step -

In [4]:
# resultados
# AUC y F1 sobre los test sets de D06-D20 (Tabla IV, AE-3).
# Comparativa réplica vs. valores reportados en el paper.

# Valores de referencia — Tabla IV del paper (columna AE-3)
PAPER = {
    ("D06", "DT"):  (94.99, 93.24), ("D06", "kNN"): (97.84, 93.98), ("D06", "SVM"): (97.45, 92.74),
    ("D09", "DT"):  (87.89, 73.88), ("D09", "kNN"): (87.28, 74.47), ("D09", "SVM"): (84.17, 74.04),
    ("D13", "DT"):  (99.98, 99.73), ("D13", "kNN"): (99.87, 99.46), ("D13", "SVM"): (98.98, 99.04),
    ("D18", "DT"):  (90.51, 93.24), ("D18", "kNN"): (91.79, 94.30), ("D18", "SVM"): (95.09, 94.77),
    ("D20", "DT"):  (96.34, 95.24), ("D20", "kNN"): (97.07, 92.68), ("D20", "SVM"): (95.12, 92.68),
}

SEP = "=" * 78
HDR = f"{'DS':<5} {'Malware':<16} {'Alg':<5} {'AUC_paper':>10} {'AUC_rep':>8} {'ΔAUC':>7}  {'F1_paper':>9} {'F1_rep':>7} {'ΔF1':>6}"
print(SEP)
print("TABLA IV — Escenario 2 (AE-3) | Réplica vs. Paper")
print(SEP)
print(HDR)
print("-" * 78)

for ds_id in TEST_IDS:
    data    = test_sets[ds_id]
    X_sc    = data["X_scaled"]
    y_test  = data["y"]
    malware = data["malware"]

    if len(np.unique(y_test)) < 2:
        print(f"  {ds_id:<5} {malware:<16} — solo una clase, métricas no computables")
        continue

    Z_test = encoder.predict(X_sc, verbose=0)

    for clf_name, clf in classifiers.items():
        y_pred = clf.predict(Z_test)
        f1     = f1_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0) * 100
        try:
            auc = roc_auc_score(y_test, clf.predict_proba(Z_test)[:, 1]) * 100
        except Exception:
            auc = float('nan')

        auc_p, f1_p = PAPER.get((ds_id, clf_name), (float('nan'), float('nan')))
        d_auc = auc - auc_p
        d_f1  = f1  - f1_p

        print(f"  {ds_id:<5} {malware:<16} {clf_name:<5} "
              f"{auc_p:>10.2f} {auc:>8.2f} {d_auc:>+7.2f}  "
              f"{f1_p:>9.2f} {f1:>7.2f} {d_f1:>+6.2f}")

print(SEP)


TABLA IV — Escenario 2 (AE-3) | Réplica vs. Paper
DS    Malware          Alg    AUC_paper  AUC_rep    ΔAUC   F1_paper  F1_rep    ΔF1
------------------------------------------------------------------------------
  D06   Kenjiro          DT         94.99    99.99   +5.00      93.24   99.99  +6.75
  D06   Kenjiro          kNN        97.84    99.99   +2.15      93.98   99.99  +6.01
  D06   Kenjiro          SVM        97.45    66.04  -31.41      92.74   99.97  +7.23
  D09   Kenjiro          DT         87.89    81.04   -6.85      73.88   52.63 -21.25
  D09   Kenjiro          kNN        87.28   100.00  +12.72      74.47  100.00 +25.53
  D09   Kenjiro          SVM        84.17    62.50  -21.67      74.04   18.87 -55.17
  D13   IRCBot           DT         99.98    98.80   -1.18      99.73   99.37  -0.36
  D13   IRCBot           kNN        99.87    99.12   -0.75      99.46   99.43  -0.03
  D13   IRCBot           SVM        98.98    66.53  -32.45      99.04   76.09 -22.95
  D18   Mirai          